In [1]:
# -*- coding: utf-8 -*-
"""
Created on Thu Jul 25 16:41:40 2024

@author: Admin
"""

#Get Master JSON from Google Document AI - Invoice Parser

from google.cloud import documentai_v1 as documentai
import os
import json
import csv
from google.oauth2 import service_account

# Set your Google Cloud project ID and processor ID
project_id = 'invoice-scanner-429009'
location = 'us'  # e.g., 'us'
processor_id = '19d2418d93446091'

key = service_account.Credentials.from_service_account_file('invoice-scanner-429009-0055d5568275.json')

# Create a Document AI client
client = documentai.DocumentProcessorServiceClient(credentials=key)

# Define the processor resource name
processor_name = f'projects/{project_id}/locations/{location}/processors/{processor_id}'

def process_document(file_path):
    try:
        # Read the file into memory
        with open(file_path, 'rb') as file:
            content = file.read()

        # Configure the process request
        request = documentai.types.ProcessRequest(
            name=processor_name,
            raw_document=documentai.types.RawDocument(
                content=content,
                mime_type='image/jpeg'  # Correct MIME type for JPEG images
            )
        )

        # Process the document
        result = client.process_document(request=request)
        return result
    except Exception as e:
        print(f"Error processing document {file_path}: {e}")
        return None

def extract_json_data(document):
    # Convert the structured data to JSON format
    json_data = documentai.Document.to_json(document)
    return json.loads(json_data)

# Directory containing the invoices
invoices_directory = 'CTE 1//'

# Process each invoice in the directory and output JSON
for filename in os.listdir(invoices_directory):
    if filename.endswith(('.jpeg', '.jpg','.JPEG')):  # Adjust the condition as needed
        
        # Define the output JSON file name
#        filename=filename.replace("Vasai-Virar_Jainam Electric And Hardware Store","V-V_JEAHS")
        output_json_name = os.path.join(invoices_directory, f"output_{filename}.json")
        
        # Check if the JSON for this invoice has already been generated
        if os.path.exists(output_json_name):
            print(f'Skipping {filename}, JSON already exists.')
            continue

        # Proceed to process the invoice
        file_path = os.path.join(invoices_directory, filename)
        result = process_document(file_path)

        if result:
            json_data = extract_json_data(result.document)

            # Save the JSON result to a file
            with open(output_json_name, 'w', encoding='utf-8') as output_file:
                json.dump(json_data, output_file, indent=4)

            print(f'Processed and saved JSON for: {filename}')
        else:
            print(f'Failed to process: {filename}')

#%%

# Extract relevant json info to csv

def extract_information(json_content, filename):
    invoice_data = {"file_name": filename}
    line_items = []

    # Dictionary to keep track of the highest confidence entity for each type
    highest_confidence_entities = {}

    # First, process invoice-level entities
    for entity in json_content.get("entities", []):
        entity_type = entity.get("type", "")
        mention_text = entity.get("mentionText", "")
        confidence = entity.get("confidence", 0.0)  # Default to 0.0 if 'confidence' is missing

        if entity_type == "line_item":
            # Process line item properties
            line_item = {"file_name": filename}
            highest_confidence_props = {}

            for attr in entity.get("properties", []):
                attr_type = attr.get("type", "")
                attr_value = attr.get("mentionText", "")
                attr_confidence = attr.get("confidence", 0.0)  # Default to 0.0 if 'confidence' is missing

                # Keep the property with the highest confidence
                if attr_type not in highest_confidence_props or attr_confidence > highest_confidence_props[attr_type].get(f"{attr_type}_confidence", 0.0):
                    prop_data = {
                        attr_type: attr_value,
                        f"{attr_type}_confidence": attr_confidence,
                        f"{attr_type}_bounding_box": extract_bounding_box(attr)
                    }
                    highest_confidence_props[attr_type] = prop_data

            # Combine all highest confidence properties into the line item
            for prop in highest_confidence_props.values():
                line_item.update(prop)

            # Add line item-level bounding box and confidence
            line_item["line_item_bounding_box"] = extract_bounding_box(entity)
            line_item["line_item_confidence"] = confidence

            line_items.append(line_item)
        else:
            # For invoice-level entities, keep the one with the highest confidence
            if entity_type not in highest_confidence_entities or confidence > highest_confidence_entities[entity_type].get(f"{entity_type}_confidence", 0.0):
                entity_data = {
                    entity_type: mention_text,
                    f"{entity_type}_confidence": confidence,
                    f"{entity_type}_bounding_box": extract_bounding_box(entity)
                }
                highest_confidence_entities[entity_type] = entity_data

    # Combine invoice-level data
    invoice_data.update({k: v for entity in highest_confidence_entities.values() for k, v in entity.items()})

    if not line_items:
        # If no line items were extracted, return a list with invoice_data only
        return [invoice_data]
    else:
        # Attach invoice-level data to each line item
        for line_item in line_items:
            line_item.update(invoice_data)
        return line_items


def extract_bounding_box(entity_or_attr):
    bounding_box = ""
    if "pageAnchor" in entity_or_attr:
        page_refs = entity_or_attr["pageAnchor"].get("pageRefs", [])
        if page_refs and "boundingPoly" in page_refs[0]:
            vertices = page_refs[0]["boundingPoly"].get("normalizedVertices", [])
            coordinates = ";".join([f"({v.get('x',0)},{v.get('y',0)})" for v in vertices])
            bounding_box = coordinates
    return bounding_box


def process_json_files(directory):
    data = []

    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    json_content = json.load(file)
                    extracted_data = extract_information(json_content, filename)

                    if not extracted_data:
                        # If nothing was extracted, add an entry with only the file name
                        data.append({"file_name": filename})
                    else:
                        data.extend(extracted_data)

            except Exception as e:
                print(f"Error processing file {filename}: {e}")
                # Optionally, you can add an entry for files that caused an error
                data.append({"file_name": filename, "error": str(e)})

    return data


def save_to_csv(data, output_file):
    if not data:
        return

    # Collect all possible keys from the data
    keys = set()
    for item in data:
        keys.update(item.keys())

    keys = list(keys)  # Convert to list

    # Optionally, you can sort the keys to have a consistent column order
    keys.sort()

    with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=keys)
        writer.writeheader()
        writer.writerows(data)


# Update the file paths accordingly
file_path = 'CTE 1//'
output_csv = os.path.join(file_path, 'output_highest_confidence_all_files.csv')

data = process_json_files(file_path)
save_to_csv(data, output_csv)

print(f"Data has been written to {output_csv}")

Processed and saved JSON for: 23mar26_Bihar_Patna_Sagar Traders_70010790_nan_69c0f6814fadd6f31703e68c_CustomFile_177425359528457901@page15.jpeg
Processed and saved JSON for: 23mar26_Karnataka_Bangalore Metropolitan Region_Sri Maruthi Paints & Hardware_51779765_nan_69c0d82f4fadd6f31703e30e_CustomFile_177423862719271970@page1.jpeg
Processed and saved JSON for: 23mar26_Rajasthan_Gangapur City_Welcome Paints Store_51583800_nan_69c125e84fadd6f31703edb9_CustomFile_177426572733871580@page1.jpeg
Processed and saved JSON for: 23mar26_Rajasthan_Gangapur City_Welcome Paints Store_51583800_nan_69c125e84fadd6f31703edb9_CustomFile_17742657273972160@page1.jpeg
Processed and saved JSON for: 23mar26_Rajasthan_Gangapur City_Welcome Paints Store_51583800_nan_69c125e84fadd6f31703edb9_CustomFile_177426572741114930@page1.jpeg
Processed and saved JSON for: 23mar26_Rajasthan_Lachhmangarh_Vimal Building Meterial_51621023_nan_69c12a0f4fadd6f31703ef8c_PHOTO_177397997492274030.JPEG
Processed and saved JSON for: 2